<a href="https://colab.research.google.com/github/Amoyeola/Hello-World/blob/master/ollama_cyber_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to build a free LLM cybersecurity lab with Google Colab and Ollama

Learn how to set up a free cybersecurity lab using Large Language Models (LLMs), Google Colab, and Ollama. Explore practical use cases for AI in cybersecurity.

## Introduction

In this notebook, we will explore the powerful combination of Large Language Models (LLMs) and cybersecurity using [Ollama](https://ollama.com/). Ollama allows us to download and run LLMs locally, providing several advantages for cybersecurity applications:

1. Data privacy: Process sensitive information without sending it to external servers.
2. Customization: Easily fine-tune models for specific cybersecurity tasks.
3. Offline capability: Perform analysis without an internet connection.
4. Cost-effective: Avoid usage fees associated with cloud-based LLM services.

We'll demonstrate two practical use-cases:
1. Generating malware information cards
2. Creating a cybersecurity news digest

While we're using Google Colab for this demonstration due to its accessibility, keep in mind some limitations:
- Runtime limits (sessions typically disconnect after 12 hours)
- Variability in GPU availability
- Potential network bandwidth constraints

**For production use, consider running this notebook on a dedicated machine or cloud instance with more stable resources.**

Let's dive in and see how we can leverage LLMs for cybersecurity tasks!

## Installation of ollama

In [1]:
!apt-get update && apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,164 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,002 kB]
Fetched 11.3 MB in 3s (4,116 kB/s)
Reading packa

In [2]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:07 --:--:--     0


In [3]:
!pkill -f ollama
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5

In [4]:
!curl --max-time 5 http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3:latest","model":"llama3:latest","modified_at":"2026-03-17T13:35:07.546705488Z","size":4661224676,"digest":"365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_0"}}]}

In [5]:
!ollama pull llama3


In [6]:
!ollama run llama3 "Say hello in one sentence."

Hello there!



Ollama is now installed. Ollama is an open-source tool that simplifies running large language models locally.
It provides an easy way to download, run, and manage various LLMs on your machine.

If you encounter any GPU-related issues, try the following troubleshooting steps:
1. Ensure CUDA is properly installed and configured.
2. Check if the correct GPU drivers are installed.
3. Verify that Ollama has access to the GPU resources.

Now, let's start Ollama in a separate thread so we can use it throughout this notebook.

We then start ollama in a separate thread so that we can use it after

Reference: https://stackoverflow.com/a/77828874

In [7]:
import os
import asyncio
import threading

# NB: You may need to set these depending and get cuda working depending which backend you are running.
# Set environment variable for NVIDIA library
# Set environment variables for CUDA
os.environ['PATH'] += ':/usr/local/cuda/bin'
# Set LD_LIBRARY_PATH to include both /usr/lib64-nvidia and CUDA lib directories
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64'

async def run_process(cmd, stdout=None, stderr=None):
    print('>>> starting', *cmd)
    process = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=stdout or asyncio.subprocess.PIPE,
        stderr=stderr or asyncio.subprocess.PIPE
    )

    if stdout is None and stderr is None:
        async def pipe(lines):
            async for line in lines:
                print(line.decode().strip())

        await asyncio.gather(
            pipe(process.stdout),
            pipe(process.stderr),
        )
    else:
        await process.wait()

async def start_ollama_serve():
    await run_process(['ollama', 'serve'],
                      stdout=open(os.devnull, 'w'),
                      stderr=open(os.devnull, 'w'))

def run_async_in_thread(loop, coro):
    asyncio.set_event_loop(loop)
    loop.run_until_complete(coro)
    loop.close()

# Create a new event loop that will run in a new thread
new_loop = asyncio.new_event_loop()

# Start ollama serve in a separate thread so the cell won't block execution
thread = threading.Thread(target=run_async_in_thread, args=(new_loop, start_ollama_serve()))
thread.start()

# Wait 5s for ollama to load
import time
time.sleep(5)

>>> starting ollama serve


## Playing with ollama API

In [8]:
# We first need to install dependencies
!pip install openai pydantic instructor

For this first example, we will use [gemma2](https://blog.google/technology/developers/google-gemma-2/) from Google in its 9 billion parameters version. Its currently [one of the best small](https://chat.lmsys.org/?leaderboard) (<12B parameters) and opensource model.


In [9]:
!ollama pull gemma2

In [10]:
MODEL = "gemma2"

In [11]:
!ollama pull gemma2
!ollama run gemma2 "Say hello in one sentence."


Hello there! 👋  




## Use Case I: Generating malware information cards

In cybersecurity, having quick access to malware information is crucial for threat analysis and incident response.
We'll use our LLM to generate concise "malware cards" based on malware names. This demonstrates how LLMs can be
used to quickly summarize and present complex security information.

However, it's important to note:
1. This information is generated based on the LLM's training data and may not always be up-to-date or accurate.
2. **Always verify critical information with authoritative sources before making security decisions**.
3. Be cautious about potential hallucinations, especially for lesser-known malware.

In [12]:
!pip install openai pydantic instructor jsonref

#### Fetch malware data

In [13]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

import instructor


class Malware(BaseModel):
    name: str
    first_seen: datetime
    language: str
    architecture: str
    developer: str
    description: str

def get_malware_description(malware_name: str) -> Malware:
  client = instructor.from_openai(
      OpenAI(
          base_url="http://127.0.0.1:11434/v1",
          api_key="ollama",  # required, but unused
      ),
      mode=instructor.Mode.JSON,
  )

  resp = client.chat.completions.create(
      model=MODEL,
      messages=[
          {
              "role": "user",
              "content": malware_name,
          }
      ],
      response_model=Malware,
  )
  return resp

malwares = ["Wannacry", "Stuxnet", "Darkcomet"]
for malware in malwares:
  resp = get_malware_description(malware)
  print(resp.model_dump_json(indent=2))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


{
  "name": "WannaCry",
  "first_seen": "2017-05-12T00:00:00Z",
  "language": "C++",
  "architecture": "x86",
  "developer": "Lazarus Group",
  "description": "A ransomware worm that spreads using EternalBlue exploit to encrypt files and demand payment for decryption."
}
{
  "name": "Stuxnet",
  "first_seen": "2010-06-17T00:00:00Z",
  "language": "C/Assembly",
  "architecture": "Windows x86",
  "developer": "Unknown (Attribution: Israel & USA)",
  "description": "A sophisticated computer worm designed to sabotage Iranian nuclear facilities."
}
{
  "name": "DarkComet",
  "first_seen": "2013-01-01T00:00:00",
  "language": "C++",
  "architecture": "32-bit, 64-bit",
  "developer": "Unknown (likely cybercrime group)",
  "description": "DarkComet is a powerful remote access Trojan (RAT). It allows attackers to control compromised computers remotely, steal data, take screenshots, record audio and video, and more."
}


## Use Case II: Building a daily cybersecurity news digest

Staying informed about the latest cybersecurity developments is essential but time-consuming. This use case
demonstrates how to leverage LLMs to create a personalized, AI-curated news digest.

Benefits:
1. Time-saving: Quickly get an overview of the most important cybersecurity news.
2. Prioritization: Focus on high-impact stories first.
3. Contextual understanding: Get AI-generated summaries and potential implications of each story.

The process involves:
1. Fetching articles from trusted RSS feeds
2. Using the LLM to rank and summarize the articles
3. Generating a comprehensive digest with actionable insights

Let's walk through each step:

In [14]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=620abc20c583ebbfdc6083910465cc706d05f7caba665434601507109e075a14
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


#### Fetching of RSS entries

In [15]:
from pydantic import BaseModel, Field
from datetime import datetime
import uuid

class Article(BaseModel):
    id: str = Field(..., description="Unique identifier for the article")
    title: str = Field(..., description="Title of the article")
    link: str = Field(..., description="URL of the article")
    published: datetime = Field(..., description="Date and time the article was published")

import feedparser
import uuid

RSS_FEEDS = [
    "https://feeds.feedburner.com/TheHackersNews",
    "https://www.bleepingcomputer.com/feed/",
]

# fetch last rss entries and return title, url and id
def fetch_rss_entries(url):
    feed = feedparser.parse(url)
    return [Article(id=uuid.uuid4().hex[:8],
                   title=entry.title,
                   link=entry.link,
                   published=datetime(*entry.published_parsed[:6]))
            for entry in feed.entries]

articles = []
for rss_feed in RSS_FEEDS:
  articles.extend(fetch_rss_entries(rss_feed))
articles

[Article(id='a2b9b6c6', title="AI is Everywhere, But CISOs are Still Securing It with Yesterday's Skills and Tools, Study Finds", link='https://thehackernews.com/2026/03/ai-is-everywhere-but-cisos-are-still.html', published=datetime.datetime(2026, 3, 17, 11, 30)),
 Article(id='8d3831cf', title='Konni Deploys EndRAT Through Phishing, Uses KakaoTalk to Propagate Malware', link='https://thehackernews.com/2026/03/konni-deploys-endrat-through-spear.html', published=datetime.datetime(2026, 3, 17, 9, 53)),
 Article(id='94342d70', title='CISA Flags Actively Exploited Wing FTP Vulnerability Leaking Server Paths', link='https://thehackernews.com/2026/03/cisa-flags-actively-exploited-wing-ftp.html', published=datetime.datetime(2026, 3, 17, 5, 23)),
 Article(id='6b4e1760', title='GlassWorm Attack Uses Stolen GitHub Tokens to Force-Push Malware Into Python Repos', link='https://thehackernews.com/2026/03/glassworm-attack-uses-stolen-github.html', published=datetime.datetime(2026, 3, 16, 19, 37)),
 A

#### Ranking using LLM

In [16]:
from pydantic import BaseModel, Field
from typing import List
import instructor
from openai import OpenAI

class ArticleRef(BaseModel):
    id: str
    relevance_score: int = Field(..., ge=1, le=10, description="Relevance score from 1-10")

class RankedCyberNews(BaseModel):
    title: str = Field(..., max_length=100, description="Concise title of the cybersecurity news")
    summary: str = Field(..., max_length=250, description="Brief summary of the news")
    impact_score: int = Field(..., ge=1, le=10, description="Impact score from 1-10")
    related_articles: List[ArticleRef] = Field(..., min_items=1, description="List of related article IDs with relevance scores")

def rank_cyber_news(articles: List[Article]) -> List[RankedCyberNews]:
    client = instructor.from_openai(
        OpenAI(
            base_url="http://127.0.0.1:11434/v1",
            api_key="ollama",  # required, but unused
        ),
        mode=instructor.Mode.JSON,
    )

    article_data = "\n".join([f"{a.id} - {a.title}" for a in articles])
    prompt = f"""As a senior cybersecurity analyst, analyze the following articles and identify the 3 most significant cybersecurity news items. For each news item:

1. Provide a concise title (max 100 characters).
2. Write a brief summary (max 250 characters).
3. Assign an impact score (1-10) based on its potential consequences for individuals, organizations, or global cybersecurity.
4. List related article IDs with a relevance score (1-10) for each.

Consider factors such as:
- The scale and severity of any breaches or vulnerabilities
- The prominence of affected organizations or systems
- The novelty or sophistication of attack methods
- Potential long-term implications for cybersecurity practices

Articles to analyze:
{article_data}

Provide your analysis in a structured format suitable for JSON parsing.
"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_model=List[RankedCyberNews],
    )
    return resp

ranked_news = rank_cyber_news(articles)
ranked_news

/tmp/ipykernel_27698/3977483455.py:14: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  related_articles: List[ArticleRef] = Field(..., min_items=1, description="List of related article IDs with relevance scores")


[RankedCyberNews(title='GlassWorm Supply Chain Attack Targets Developers', summary='The GlassWorm hacking group exploits 72 open VSX extensions to compromise developers. This attack chain leverages stolen GitHub tokens and pushes malware into Python repositories.', impact_score=9, related_articles=[ArticleRef(id='6b4e1760', relevance_score=10), ArticleRef(id='2b3579d3', relevance_score=8)]),
 RankedCyberNews(title='Exploiting n8n for Server Takeover and Data Leakage', summary='Critical vulnerabilities in n8n allow attackers to gain remote code execution (RCE) and steal stored credentials. This highlights the risks of relying on open-source tools without proper security audits.', impact_score=8, related_articles=[ArticleRef(id='21ede9d4', relevance_score=10), ArticleRef(id='fd622ae2', relevance_score=9)]),
 RankedCyberNews(title='DRILLAPP Backdoor Targets Ukraine with Microsoft Edge Espionage', summary="Ukrainian systems are targeted with a sophisticated backdoor, DRILLAPP, abusing Micr

In [17]:
# For each subject, we will parse articles using newspaper lib and ask for an actionnable summary
!pip install newspaper4k

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 6.0.2
    Uninstalling lxml-6.0.2:
      Successfully uninstalled lxml-6.0.2


#### Article scraping

In [18]:
from newspaper import Article as ArticleParser

full_text = ""

for ranked_article in ranked_news:
    full_text += f"# News: {ranked_article.title}\n\n"
    full_text += f"Summary: {ranked_article.summary}\n\n"
    full_text += f"Impact Score: {ranked_article.impact_score}\n\n"

    # Select the article with the highest relevance score
    if ranked_article.related_articles:
        most_relevant_article = max(ranked_article.related_articles, key=lambda x: x.relevance_score)

        # Get article link using id
        article_link = next((a.link for a in articles if a.id == most_relevant_article.id), None)
        if article_link:
            full_text += f"# Most Relevant Article (Relevance: {most_relevant_article.relevance_score}):\n\n{article_link}\n\n"
            try:
                article_obj = ArticleParser(article_link)
                article_obj.download()
                article_obj.parse()
                full_text += f"# Text:\n\n{article_obj.text}\n\n"  # Limit to first 1000 characters
            except Exception as e:
                full_text += f"Error fetching article: {str(e)}\n\n"
    else:
        full_text += "No related articles found for this news item.\n\n"

print(f"Processed {len(ranked_news)} news items.")

Processed 3 news items.


#### Digest building

In [24]:
from openai import OpenAI

def summarize_articles(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    As an expert cybersecurity analyst, create a concise yet comprehensive daily news digest in markdown format. Your task:

    1. Analyze the provided cybersecurity articles.
    2. Summarize the given stories, which are already ranked by importance.
    3. For each story, include:
       - The provided headline
       - A brief summary (2-3 sentences) highlighting key points
       - The potential impact on individuals or organizations (consider the given impact score)
       - Key details from the most relevant related article
       - Actionable advice or takeaways for readers

    4. Use bullet points for clarity and readability.
    5. Conclude with a "Bottom Line" that provides an overall assessment of the day's cybersecurity landscape.

    Guidelines:
    - Use clear, professional language avoiding unnecessary jargon.
    - Prioritize stories based on their impact scores.
    - Ensure the total digest length is between 500-750 words.
    - Use markdown formatting to enhance readability (e.g., headers, bold for emphasis).

    Articles to summarize:

    {full_text}

    Produce only the markdown-formatted digest without any additional commentary or metadata.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

digest = summarize_articles(full_text)
print(digest)

## Daily Cybersecurity Digest - March 16, 2026

**• GlassWorm Supply Chain Attack Targets Developers (Impact Score: 9)**

  * **Headline:** GlassWorm Supply Chain Attack Targets Developers
  * **Summary:** The GlassWorm hacking group exploited 72 open VSX extensions to compromise developers. This attack chain leverages stolen GitHub tokens and pushes malware into Python repositories.  Attackers use a technique called force-pushing, which rewrites git history and leaves no trace of the compromised commits on GitHub's UI.
  * **Potential Impact:** High risk for developers using compromised packages and organizations utilizing vulnerable dependencies.

    **Actionable Advice:** 

     * Vet and verify all dependencies before installation.
     * Regularly scan projects for known vulnerabilities and suspicious code.  
     * Enable multi-factor authentication (MFA) on developer accounts, including GitHub.


 **• Exploiting n8n for Server Takeover and Data Leakage (Impact Score: 8)**

   *

#### Bonus: build your daily podcast

In this part, we will use another prompt to generate the script for a podcast about the last cybersecurity news. We will then use

# New Section

In [50]:
def build_podcast(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    You are an engaging cybersecurity podcast host. Create a script for a 5-minute daily news digest based on the provided articles. Your task:

    1. Start with a brief, catchy introduction.
    2. Summarize 2-3 key cybersecurity stories from the given content.
    3. For each story, provide:
       - A concise overview of the incident or development
       - Its potential impact on individuals or organizations
       - Quick, actionable advice for listeners (if applicable)
    4. Use a conversational tone, as if speaking directly to the audience.
    5. Conclude with a brief sign-off, encouraging listeners to stay vigilant.

    Guidelines:
    - Keep the total length around 600-800 words.
    - Use simple language and avoid technical jargon when possible.
    - Include relevant statistics or numbers to add context.
    - Add brief transitions between stories to maintain flow.

    Content to summarize:

    {full_text}

    Remember, create only the podcast script without any formatting characters or annotations.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

podcast = build_podcast(full_text)
podcast

"Hey everyone, and welcome back! It's your daily cybersecurity rundown, where we break down the biggest threats and help you stay safe online.\n\nLet's kick things off with some major news in the world of open source security. The GlassWorm hacking group, known for its nasty supply chain attacks, has been caught red-handed injecting malware into hundreds of Python repositories on GitHub. They're using stolen developer credentials to rewrite code files and slip in malicious payloads. This is a serious problem because it affects everything from Django applications to AI research code. If you're developing with Python, be sure to double-check the integrity of any libraries you're using. It's vital to verify their source and download them directly from trusted sources.\n\nMoving on, our next story involves the open-source workflow automation platform, n8n. A critical vulnerability found in this software allows attackers to take complete control of systems. The good news is that a patch has

In [51]:
!git clone https://github.com/myshell-ai/MeloTTS.git
%cd MeloTTS


Cloning into 'MeloTTS'...
remote: Enumerating objects: 432, done.
remote: Total 432 (delta 0), reused 0 (delta 0), pack-reused 432 (from 1)
Receiving objects: 100% (432/432), 6.05 MiB | 12.21 MiB/s, done.
Resolving deltas: 100% (219/219), done.
/content/MeloTTS/MeloTTS/MeloTTS/MeloTTS/MeloTTS/MeloTTS


In [52]:
from melo.api import TTS
# Speed is adjustable
speed = 1.1

# CPU is sufficient for real-time inference.
# You can set it manually to 'cpu' or 'cuda' or 'cuda:0' or 'mps'
device = 'auto' # Will automatically use GPU if available

# English
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id

# Default accent
output_path = 'en-default.wav'
model.tts_to_file(podcast, speaker_ids['EN-Default'], output_path, speed=speed)

ModuleNotFoundError: No module named 'cn2an'

In [ ]:
from IPython.display import Audio #Import Audio method from IPython's Display Class

Audio(output_path, autoplay=False)